Разработать консольное приложение на языке программирования (например, C#), которое моделирует работу компании по учету заказов. Программа должна использовать классы для описания сущностей, их атрибутов и методов. Для реализации логики будут использоваться коллекции для хранения связанных данных, а также делегаты и события для автоматизации уведомлений о ключевых действиях в системе.

In [8]:
using System;
using System.Collections.Generic;
using System.Linq;

// 1. КОМПАНИЯ
// Хранит сотрудников, заказчиков и заказы, 
// выполняет добавление, поиск, фильтрацию и формирование отчетов


public class Company
{
    public string Name { get; set; }

    private List<Employee> employees = new List<Employee>();
    private List<Customer> customers = new List<Customer>();
    private List<Order> orders = new List<Order>();

    public Company(string name)
    {
        Name = name;
    }

    public void AddEmployee(Employee employee)
    {
        employees.Add(employee);
        Console.WriteLine($"Сотрудник {employee.GetFullName()} добавлен.");
    }

    public void AddCustomer(Customer customer)
    {
        customers.Add(customer);
        Console.WriteLine($"Заказчик {customer.GetFullName()} зарегистрирован.");
    }

    public void AddOrder(Order order)
    {
        orders.Add(order);
        order.StatusChanged += CompanyNotification;

        Console.WriteLine($"Заказ №{order.OrderId} принят в систему.");
    }

    public void AssignEmployee(Order order, Employee employee)
    {
        if (!orders.Contains(order))
            throw new InvalidOperationException("Такого заказа нет в системе.");

        if (!employees.Contains(employee))
            throw new InvalidOperationException("Сотрудник не зарегистрирован в компании.");

        order.AssignEmployee(employee);

        Console.WriteLine(
            $"На заказ №{order.OrderId} назначен сотрудник {employee.GetFullName()}."
        );
    }

    public void UpdateOrderStatus(Order order, OrderStatus status)
    {
        if (!orders.Contains(order))
            throw new InvalidOperationException("Заказ не найден в компании.");

        if (order.AssignedEmployee == null)
            throw new InvalidOperationException("На заказ не назначен сотрудник.");

        order.ChangeStatus(status);
    }

    private void CompanyNotification(string message)
    {
        Console.WriteLine($"[Событие компании] {message}");
    }

    public void ShowCompanyInfo()
    {
        Console.WriteLine("\n========== ИНФОРМАЦИЯ О КОМПАНИИ ==========");
        Console.WriteLine($"Название: {Name}");
        Console.WriteLine($"Сотрудников: {employees.Count}");
        Console.WriteLine($"Заказчиков: {customers.Count}");
        Console.WriteLine($"Всего заказов: {orders.Count}");
    }

    public void ShowAllOrders()
    {
        Console.WriteLine("\n========== СПИСОК ВСЕХ ЗАКАЗОВ ==========");

        foreach (Order order in orders)
        {
            Console.WriteLine("----------------------------------------");
            order.ShowInfo();
        }
    }

    public void ShowOrdersByStatus(OrderStatus status)
    {
        Console.WriteLine($"\n========== ЗАКАЗЫ: {status} ==========");

        var result = orders.Where(o => o.Status == status).ToList();

        if (result.Count == 0)
        {
            Console.WriteLine("Заказы не найдены.");
            return;
        }

        foreach (Order order in result)
        {
            Console.WriteLine("----------------------------------------");
            order.ShowInfo();
        }
    }

    public void SearchOrders(string customerName)
    {
        Console.WriteLine($"\n========== ПОИСК ЗАКАЗОВ: {customerName} ==========");

        var result = orders
            .Where(o => o.Customer.MatchesName(customerName))
            .ToList();

        if (result.Count == 0)
        {
            Console.WriteLine("По вашему запросу ничего не найдено.");
            return;
        }

        foreach (Order order in result)
        {
            Console.WriteLine("----------------------------------------");
            order.ShowInfo();
        }
    }

    public void ShowEmployeeReport()
    {
        Console.WriteLine("\n========== ОТЧЕТ ПО СОТРУДНИКАМ ==========");

        foreach (Employee employee in employees)
        {
            Console.WriteLine("----------------------------------------");
            employee.ShowInfo();
        }
    }
}


// 2. СОТРУДНИК
// Содержит данные сотрудника и методы для работы с ними

public class Employee : Person
{
    public string Position { get; set; }
    public int OrdersProcessed { get; private set; }

    public Employee(string firstName, string lastName, string position)
        : base(firstName, lastName)
    {
        Position = position;
        OrdersProcessed = 0;
    }


    public void ProcessOrder()
    {
        OrdersProcessed++;
    }

    public void Notify(string message)
    {
        Console.WriteLine($"[Уведомление для {GetFullName()}] {message}");
    }

    public override void ShowInfo()
    {
        Console.WriteLine($"Имя: {GetFullName()}");
        Console.WriteLine($"Должность: {Position}");
        Console.WriteLine($"Выполнено заказов: {OrdersProcessed}");
    }
}


// 3. ЗАКАЗЧИК
// Хранит информацию о заказчике и позволяет изменять и искать данные

public class Customer : Person
{
    public string PhoneNumber { get; set; }

    public Customer(string firstName, string lastName, string phoneNumber)
        : base(firstName, lastName)
    {
        PhoneNumber = phoneNumber;
    }

    public void ChangePhoneNumber(string newPhone)
    {
        PhoneNumber = newPhone;
        Console.WriteLine($"Телефон заказчика {GetFullName()} обновлен.");
    }

    public bool MatchesName(string name)
    {
        return GetFullName().Contains(
            name,
            StringComparison.OrdinalIgnoreCase
        );
    }

    public override void ShowInfo()
    {
        Console.WriteLine($"Имя: {GetFullName()}");
        Console.WriteLine($"Телефон: {PhoneNumber}");
    }
}

// 4. ЗАКАЗ
// Содержит описание, заказчика, ответственного, статус и дату

public enum OrderStatus
{
    Created,
    InProgress,
    Completed
}

public delegate void OrderNotification(string message);

public class Order
{
    private static int nextId = 1;

    private bool countedAsCompleted = false;

    public int OrderId { get; private set; }
    public string Description { get; set; }
    public Customer Customer { get; set; }
    public Employee AssignedEmployee { get; private set; }
    public OrderStatus Status { get; private set; }
    public DateTime CreatedAt { get; private set; }

    public event OrderNotification StatusChanged;

    public Order(string description, Customer customer)
    {
        OrderId = nextId++;
        Description = description;
        Customer = customer;
        Status = OrderStatus.Created;
        CreatedAt = DateTime.Now;
    }

    public void AssignEmployee(Employee employee)
    {
        if (employee == null)
            throw new ArgumentNullException(nameof(employee));

        if (AssignedEmployee != null)
            StatusChanged -= AssignedEmployee.Notify;

        AssignedEmployee = employee;

        StatusChanged += employee.Notify;
    }

    public void ChangeStatus(OrderStatus newStatus)
    {
        if (Status == newStatus)
            throw new InvalidOperationException("Заказ уже имеет такой статус.");

        OrderStatus oldStatus = Status;
        Status = newStatus;

        if (newStatus == OrderStatus.Completed && !countedAsCompleted)
        {
            AssignedEmployee?.ProcessOrder();
            countedAsCompleted = true;
        }

        StatusChanged?.Invoke(
            $"Заказ №{OrderId}: {oldStatus} -> {newStatus}"
        );
    }

    public void ShowInfo()
    {
        Console.WriteLine($"Заказ №{OrderId}");
        Console.WriteLine($"Описание: {Description}");
        Console.WriteLine($"Заказчик: {Customer.GetFullName()}");
        Console.WriteLine($"Телефон: {Customer.PhoneNumber}");
        Console.WriteLine($"Ответственный: {(AssignedEmployee == null ? "не назначен" : AssignedEmployee.GetFullName())}");
        Console.WriteLine($"Статус: {Status}");
        Console.WriteLine($"Дата создания: {CreatedAt:dd.MM.yyyy HH:mm:ss}");
    }
}

// БАЗОВЫЙ КЛАСС PERSON
// Используется для наследования и полиморфизма

public abstract class Person
{
    public string FirstName { get; set; }
    public string LastName { get; set; }

    protected Person(string firstName, string lastName)
    {
        FirstName = firstName;
        LastName = lastName;
    }

    public string GetFullName()
    {
        return $"{FirstName} {LastName}";
    }

    public abstract void ShowInfo();
}

try
{
    Console.WriteLine("============================================");
    Console.WriteLine("       СИСТЕМА УЧЕТА ЗАКАЗОВ КОМПАНИИ");
    Console.WriteLine("============================================");

    Company company = new Company("NovaTech Solutions");

    // Создание сотрудников
    Employee employee1 = new Employee("Алиса", "Воронова", "Руководитель проектов");
    Employee employee2 = new Employee("Максим", "Орлов", "Разработчик");
    Employee employee3 = new Employee("Виктория", "Соколова", "Аналитик");

    Console.WriteLine("\n--- РЕГИСТРАЦИЯ СОТРУДНИКОВ ---");

    company.AddEmployee(employee1);
    company.AddEmployee(employee2);
    company.AddEmployee(employee3);

    // Создание заказчиков
    Customer customer1 = new Customer("Марк", "Белозеров", "+7-901-456-78-21");
    Customer customer2 = new Customer("Полина", "Лебедева", "+7-902-345-19-63");
    Customer customer3 = new Customer("Артур", "Князев", "+7-903-789-12-45");

    Console.WriteLine("\n--- РЕГИСТРАЦИЯ ЗАКАЗЧИКОВ ---");

    company.AddCustomer(customer1);
    company.AddCustomer(customer2);
    company.AddCustomer(customer3);

    // Создание заказов
    Order order1 = new Order("Создание мобильного приложения", customer1);
    Order order2 = new Order("Разработка веб-сайта", customer2);
    Order order3 = new Order("Настройка базы данных", customer3);
    Order order4 = new Order("Разработка интернет-магазина", customer1);
    Order order5 = new Order("Создание CRM-системы", customer2);

    Console.WriteLine("\n--- ПРИЕМ ЗАКАЗОВ ---");

    company.AddOrder(order1);
    company.AddOrder(order2);
    company.AddOrder(order3);
    company.AddOrder(order4);
    company.AddOrder(order5);

    Console.WriteLine("\n--- НАЗНАЧЕНИЕ СОТРУДНИКОВ ---");

    company.AssignEmployee(order1, employee2);
    company.AssignEmployee(order2, employee1);
    company.AssignEmployee(order3, employee3);
    company.AssignEmployee(order4, employee2);
    company.AssignEmployee(order5, employee1);

    Console.WriteLine("\n--- ОБРАБОТКА ЗАКАЗОВ ---");

    company.UpdateOrderStatus(order1, OrderStatus.InProgress);
    company.UpdateOrderStatus(order2, OrderStatus.Completed);
    company.UpdateOrderStatus(order3, OrderStatus.InProgress);
    company.UpdateOrderStatus(order4, OrderStatus.Completed);
    company.UpdateOrderStatus(order5, OrderStatus.InProgress);

    Console.WriteLine("\n--- ПРОВЕРКА ОБРАБОТКИ ОШИБОК ---");

    try
    {
        company.UpdateOrderStatus(order2, OrderStatus.Completed);
    }
    catch (InvalidOperationException ex)
    {
        Console.WriteLine($"Ошибка обработки: {ex.Message}");
    }

    company.ShowCompanyInfo();
    company.ShowAllOrders();

    company.ShowOrdersByStatus(OrderStatus.InProgress);
    company.ShowOrdersByStatus(OrderStatus.Completed);

    company.SearchOrders("Марк");
}
catch (Exception ex)
{
    Console.WriteLine($"\nКритическая ошибка: {ex.Message}");
}


       СИСТЕМА УЧЕТА ЗАКАЗОВ КОМПАНИИ

--- РЕГИСТРАЦИЯ СОТРУДНИКОВ ---
Сотрудник Алиса Воронова добавлен.
Сотрудник Максим Орлов добавлен.
Сотрудник Виктория Соколова добавлен.

--- РЕГИСТРАЦИЯ ЗАКАЗЧИКОВ ---
Заказчик Марк Белозеров зарегистрирован.
Заказчик Полина Лебедева зарегистрирован.
Заказчик Артур Князев зарегистрирован.

--- ПРИЕМ ЗАКАЗОВ ---
Заказ №1 принят в систему.
Заказ №2 принят в систему.
Заказ №3 принят в систему.
Заказ №4 принят в систему.
Заказ №5 принят в систему.

--- НАЗНАЧЕНИЕ СОТРУДНИКОВ ---
На заказ №1 назначен сотрудник Максим Орлов.
На заказ №2 назначен сотрудник Алиса Воронова.
На заказ №3 назначен сотрудник Виктория Соколова.
На заказ №4 назначен сотрудник Максим Орлов.
На заказ №5 назначен сотрудник Алиса Воронова.

--- ОБРАБОТКА ЗАКАЗОВ ---
[Событие компании] Заказ №1: Created -> InProgress
[Уведомление для Максим Орлов] Заказ №1: Created -> InProgress
[Событие компании] Заказ №2: Created -> Completed
[Уведомление для Алиса Воронова] Заказ №2: Created -> 